In [ ]:
# Step -1 Download the data from the link shared in the Chat
# Step -2 Upload the data in Google Colab File Section
# Step -3 Copy the path of the file, create a variable path and paste
path = '/content/order.pdf'

In [ ]:
## Step -4 Installation of Libraries
#### Libraries to get installed and also to put the names in requirements.txt
!pip install tabula-py pandas sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.3 MB/s eta 0:00:00


In [ ]:
### Step 5 Create an Google GEMINI API Key -- Copy That API key
### Step 6 Click on the Secrets in Google Colab and Paste the API key value and give a name and also activate the toggle

In [ ]:
### Step -7 Import of packages
import tabula
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
## Step -8
##### Load the GEMINI API KEY from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if GEMINI_API_KEY is None:
  raise ValueError("No GEMINI_API_KEY found in Colab Secrets")
### Configuring the API Key to be used
genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
## Step 9
### Extracting the tables from the Pdf
pdf_path = '/content/order.pdf'
tables = tabula.read_pdf(pdf_path, pages='all')
df = pd.concat(tables, ignore_index=True)
df

,order_id,order_status,customer,order_date,order_quantity,sales
0,3,Order\rFinished,Muhammed Mac\rIntyre,13/10/2010,6,523080
1,293,Order\rFinished,Barry French,1/10/2012,49,20246040
2,483,Order\rFinished,Clay Rozendal,10/7/2011,30,9931519
3,515,Order\rFinished,Carlos Soltero,28/8/2010,19,788540
4,613,Order\rFinished,Carl Jackson,17/6/2011,12,187080
5,643,Order\rFinished,Monica Federle,24/3/2011,21,5563640
6,678,Order\rReturned,Dorothy Badders,26/2/2010,44,456820
7,807,Order\rFinished,Neola Schneider,23/11/2010,45,393700
8,868,Order\rFinished,Carlos Daly,8/6/2012,32,1433680
9,933,Order\rFinished,Claudia Miner,4/8/2012,15,161220


In [ ]:
meta_data = df.describe(include="all").fillna("")
meta_data

,order_id,order_status,customer,order_date,order_quantity,sales
count,20.0,20,20,20,20.0,20.0
unique,,2,16,19,,
top,,Order\rFinished,Carlos Soltero,4/8/2012,,
freq,,19,3,2,,
mean,1003.65,,,,27.3,4032125.35
std,514.490272,,,,13.010522,6859259.145341
min,3.0,,,,6.0,118060.0
25%,635.5,,,,15.75,342045.0
50%,964.0,,,,26.5,764750.0
75%,1443.75,,,,35.75,4114145.0


In [ ]:
### Step 10
### Converting the dataframe rows to text
documents = []
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['order_id']} has status {row['order_status']}\n"
      f"Customer is {row['customer']}."
      f"Order Date is {row['order_date']}."
      f"Quantity Ordered is {row['order_quantity']}."
      f"Total Sales amount is {row['sales']}."
  )
  documents.append(doc)

print(documents)

['Order ID: 3 has status Order\rFinished\nCustomer is Muhammed Mac\rIntyre.Order Date is 13/10/2010.Quantity Ordered is 6.Total Sales amount is 523080.', 'Order ID: 293 has status Order\rFinished\nCustomer is Barry French.Order Date is 1/10/2012.Quantity Ordered is 49.Total Sales amount is 20246040.', 'Order ID: 483 has status Order\rFinished\nCustomer is Clay Rozendal.Order Date is 10/7/2011.Quantity Ordered is 30.Total Sales amount is 9931519.', 'Order ID: 515 has status Order\rFinished\nCustomer is Carlos Soltero.Order Date is 28/8/2010.Quantity Ordered is 19.Total Sales amount is 788540.', 'Order ID: 613 has status Order\rFinished\nCustomer is Carl Jackson.Order Date is 17/6/2011.Quantity Ordered is 12.Total Sales amount is 187080.', 'Order ID: 643 has status Order\rFinished\nCustomer is Monica Federle.Order Date is 24/3/2011.Quantity Ordered is 21.Total Sales amount is 5563640.', 'Order ID: 678 has status Order\rReturned\nCustomer is Dorothy Badders.Order Date is 26/2/2010.Quantit

In [ ]:
### Create Embeddings - Step 11
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, convert_to_numpy=True, show_progress_bar=True)
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-0.0400941  -0.03813325  0.05240958 ... -0.08024698 -0.04358123
   0.05105155]
 [-0.03108111 -0.01880541 -0.00477829 ... -0.04830392 -0.05759401
  -0.01386821]
 [-0.06304454 -0.0310075   0.00952892 ... -0.03114309 -0.07441583
  -0.01718369]
 ...
 [ 0.02139985  0.00150405  0.07544634 ... -0.03270833 -0.01454123
  -0.02664857]
 [-0.09210814  0.01441543  0.04306164 ... -0.0356649  -0.00857276
   0.002139  ]
 [-0.06764054 -0.00706391  0.00825968 ... -0.00477096 -0.08604836
   0.00569128]]


In [ ]:
### Store the embeddings in FAISS  -Step -12
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
faiss.write_index(index, 'faiss_index.faiss')  #### I am saving the index file as well

In [ ]:
### Retrieval Function ### Step 13
def retrive_context(query, k=3):
  query_embedding = embedding_model.encode([query])
  distances, indices = index.search(query_embedding, k)
  return "\n".join([documents[i] for i in  indices[0]])

In [ ]:
### GEMINI Model Configuration ### Step -14
### max_output_tokens , temp (helps the model to create response  - creative or deterministic [0-1])
## The higher the temp - the higher the creativity is , lower the temp  - the more deterministic respose.
generation_config = {
    "temperature": 0.4,
    "max_output_tokens": 512
}
print(generation_config)
gemini_model = genai.GenerativeModel(model_name='models/gemini-2.5-flash',generation_config=generation_config)

{'temperature': 0.4, 'max_output_tokens': 512}


In [ ]:
### Step -15 #### Conversational Bot
chat_history = []

def chat_with_bot(user_input):
    global chat_history

    context = retrive_context(user_input)
    prompt = f"""
    You are a helpful conversational data analyst assistant. Please refer to the context below and answer the user's question.
    Context:
    {context}
    User's Question:
    {user_input}

    Rules:
    - Be Conversational
    - Answer only using the context
    - If you don't know the answer, say you don't have enough information
    """
    response = gemini_model.generate_content(prompt)
    answer = response.text
    chat_history.append({"user": user_input, "bot": answer})
    return answer

In [ ]:
#### Final Step ### Step -16
print("Order Analytics Chat Bot Ready !!!")
print("Type 'exit' to stop\n")

while True:
  user_input = input("User: ")
  if user_input.lower() in ['exit', 'quit', 'bye']:
    print("Good Bye !!!")
    break
  response = chat_with_bot(user_input)
  print(f"Bot: {response}")
  print("-"*60)

Order Analytics Chat Bot Ready !!!
Type 'exit' to stop

User: antara
Bot: It looks like your question might be incomplete or a typo. Could you please clarify what you're asking? I'd be happy to help if I understand your request better!
------------------------------------------------------------
User: patil
Bot: I'm sorry, I don't have enough information to answer your question about "patil" based on the provided context. The context contains details about specific orders, customers, dates, quantities, and sales amounts.
------------------------------------------------------------
User: test
Bot: I'm sorry, I don't have enough information to answer that question. Could you please ask a question related to the order data provided?
------------------------------------------------------------
User: sagarpatil@gmail.com
Bot: I'm sorry, but I don't have any information about "sagarpatil@gmail.com" in the context provided. I can only answer questions based on the data I have.
---------------

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3695.86ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1501.09ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3199.41ms


Bot: Based on the information provided, Jim Radford has the highest sales with a total of 1,669,808.
------------------------------------------------------------
User: quit
Good Bye !!!


In [ ]:
## PDF Extraction ----> Converted that to Dataframe ----> Converted to Text
### --- Created the Embeddings ----> Stored to Vector DB ---- > Genererated Context ----- >
#### --- Created the Model Configuration --- > Prompt Engineering ---- > Creating the Conversational Bot
#### Project Solution Architecture

#### What is API ?
### What is Pre-trained models ? Examples ?
### I have to built a gen AI application to support my sales team ? what kind of applications I can built to improve sales perf.
### What is LLM Models ?
### What is a prompt and why does prompt quality matter ?
### What are tokens in language models ?
### Tell me about different embeddings models ?
### How do transformers work ?
### What is RAG ? How do you use it in your current project ?
### What are multi-modals ?
### What are the challenges in GEN AI ?


### Next Step - Agentic AI // Langchain (Framework) ### Copilot   ### Bedrock (AWS)
### Create chatbots on any data using RAG and LLM ### # 3-4 indutry vertical put in resume
###

In [ ]:
from sentence_transformers import SentenceTransformer

# Sample documents
documents = [
    "Machine learning is amazing",
    "Python is used for data science",
    "Embeddings convert text into vectors"
]

# Load model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-0.04862164 -0.08617093  0.08960794 ...  0.07205722 -0.00438845
  -0.02266295]
 [-0.07593995 -0.00334905 -0.05362206 ...  0.09480739  0.1315781
  -0.0047854 ]
 [ 0.01129194  0.01835519 -0.02497821 ...  0.04084289  0.08770176
  -0.04275898]]


In [ ]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.4 MB/s eta 0:00:00


In [ ]:
import faiss

### Store the embeddings in FAISS - Step 12

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

faiss.write_index(index, 'faiss_index.faiss')

print("FAISS index saved successfully")

FAISS index saved successfully


In [ ]:
### Retrieval Function ### Step 13
def retrive_context(query, k=3):
  query_embedding = embedding_model.encode([query])
  distances, indices = index.search(query_embedding, k)
  return "\n".join([documents[i] for i in  indices[0]])

In [ ]:
### GEMINI Model Configuration ### Step -14
### max_output_tokens , temp (helps the model to create response  - creative or deterministic [0-1])
## The higher the temp - the higher the creativity is , lower the temp  - the more deterministic respose.
generation_config = {
    "temperature": 0.4,
    "max_output_tokens": 512
}
print(generation_config)
gemini_model = genai.GenerativeModel(model_name='models/gemini-2.5-flash',generation_config=generation_config)

{'temperature': 0.4, 'max_output_tokens': 512}


NameError: name 'genai' is not defined

In [ ]:
### Step -15 #### Conversational Bot
chat_history = []

def chat_with_bot(user_input):
    global chat_history

    context = retrive_context(user_input)
    prompt = f"""
    You are a helpful conversational data analyst assistant. Please refer to the context below and answer the user's question.
    Context:
    {context}
    User's Question:
    {user_input}

    Rules:
    - Be Conversational
    - Answer only using the context
    - If you don't know the answer, say you don't have enough information
    """
    response = gemini_model.generate_content(prompt)
    answer = response.text
    chat_history.append({"user": user_input, "bot": answer})
    return answer

In [ ]:
# ================================
# FINAL CHAT LOOP
# ================================

print("\n===================================")
print("Order Analytics Chat Bot Ready !!!")
print("Type 'exit' to stop")
print("===================================\n")

while True:

    user_input = input("User: ")

    if user_input.lower() in ['exit', 'quit', 'bye']:

        print("Good Bye !!!")

        break

    response = chat_with_bot(user_input)

    print(f"\nBot: {response}")

    print("-" * 60)


Order Analytics Chat Bot Ready !!!
Type 'exit' to stop

User: what is total sales


NameError: name 'gemini_model' is not defined

In [ ]:
!pip install -q google-genai sentence-transformers faiss-cpu pandas numpy

In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

from google import genai

# =========================================
# CONFIGURE GEMINI API
# =========================================

API_KEY = "GEMINI_API_KEY"

client = genai.Client(api_key=API_KEY)

# =========================================
# SAMPLE DATA
# =========================================

data = {
    "text": [
        "Order 101 was delivered successfully",
        "Order 102 is delayed due to weather conditions",
        "Customer requested refund for Order 103",
        "Order 104 is currently in transit",
        "Order 105 payment failed during checkout"
    ]
}

df = pd.DataFrame(data)

documents = df["text"].tolist()

print(documents)

# =========================================
# CREATE EMBEDDINGS
# =========================================

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

# =========================================
# CREATE FAISS INDEX
# =========================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

faiss.write_index(
    index,
    "faiss_index.faiss"
)

print("FAISS Index Saved Successfully")

# =========================================
# SEARCH DOCUMENTS
# =========================================

def search_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

# =========================================
# CHATBOT FUNCTION
# =========================================

chat_history = []

def chat_with_bot(user_input):

    relevant_docs = search_documents(user_input)

    context = "\n".join(relevant_docs)

    prompt = f"""
    You are an Order Analytics Chatbot.

    Context:
    {context}

    User Question:
    {user_input}

    Rules:
    - Give short answers
    - If answer not found, say:
      I don't have enough information.
    """

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )

    answer = response.text

    chat_history.append({
        "user": user_input,
        "bot": answer
    })

    return answer

# =========================================
# CHAT LOOP
# =========================================

print("\n===================================")
print("Order Analytics Chat Bot Ready !!!")
print("Type 'exit' to stop")
print("===================================\n")

while True:

    user_input = input("User: ")

    if user_input.lower() in [
        "exit",
        "quit",
        "bye"
    ]:

        print("Good Bye !!!")

        break

    response = chat_with_bot(user_input)

    print(f"\nBot: {response}")

    print("-" * 60)

['Order 101 was delivered successfully', 'Order 102 is delayed due to weather conditions', 'Customer requested refund for Order 103', 'Order 104 is currently in transit', 'Order 105 payment failed during checkout']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape: (5, 384)
FAISS Index Saved Successfully

Order Analytics Chat Bot Ready !!!
Type 'exit' to stop

User: what is total sales


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}